In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langfuse.langchain import CallbackHandler

load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
langfuse_trace = CallbackHandler()

llm=init_chat_model(model='claude-sonnet-5', model_provider='anthropic')
# llm=init_chat_model(model='claude-haiku-4-5-20251001', model_provider='anthropic')
ollama_llm = init_chat_model(
    model="granite4",
    model_provider="ollama"
)

llm
ollama_llm

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings

# embeddings=OpenAIEmbeddings()
embeddings=OllamaEmbeddings(
    model="nomic-embed-text"
)
embeddings

In [ ]:
## Document Ingestion And Processing
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict

In [ ]:
# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()
docs

In [ ]:
## chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
all_splits

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=all_splits,
    embedding=embeddings,
)

print(f"Vector store created with {vector_store.index.ntotal} vectors")

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool()
def retrieve(query: str):
    """Retrieve the information related to the query"""

    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return (serialized, retrieved_docs)

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# Step 1: Generate an AIMessage that may include a tool-call to be sent.
def query_or_respond(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"], config={"callbacks": [langfuse_trace]})
    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}

In [ ]:
# Step 2: Execute the retrieval.
tools = ToolNode([retrieve])
tools

In [ ]:
# Step 3: Generate a response using the retrieved content.
def generate(state: MessagesState):
    """Generate answer."""
    # Get generated ToolMessages
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    system_message_content = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        "\n\n"
        f"{docs_content}"
    )
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    # Run
    response = llm.invoke(prompt, config={"callbacks": [langfuse_trace]})
    return {"messages": [response]}

In [ ]:
# Build graph
graph_builder = StateGraph(MessagesState)

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
graph

In [ ]:
# Specify an ID for the thread
config = {"configurable": {"thread_id": "abc123"}, "callbacks": [langfuse_trace]}

In [ ]:
input_message="Hello"
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()
    

In [ ]:
vector_store.similarity_search("What is TAsk Decomposition")

In [ ]:
input_message = "What is Task Decomposition?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config,
):
    step["messages"][-1].pretty_print()

In [ ]:
### Conversation History
chat_history=graph.get_state(config).values["messages"]
for message in chat_history:
    message.pretty_print()

### ReAct Agent Architecture-Persistant Memory

In [ ]:
retrieve

In [ ]:
from langchain.agents import create_agent

memory = MemorySaver()
agent_executor=create_agent(llm,[retrieve],checkpointer=memory)

In [ ]:
agent_executor

In [ ]:
config = {"configurable": {"thread_id": "def234"}, "callbacks": [langfuse_trace]}

In [ ]:
for event in agent_executor.stream({"messages": [{"role": "user", "content": input_message}]},
                                   stream_mode="values",
                                   config=config):
    event["messages"][-1].pretty_print()